# L4: Large language models

<div class="alert alert-warning">

⚠️ The models in this lab **require a GPU with at least 10–12GB VRAM.**  Unless you have such a GPU, you **will not be able to run this lab on your own computer**.  You can access the GPU-equipped lab rooms in B-huset – Åsgard, Egypten, Olympen – to work on this lab, preferably during the times we have booked one or more of these rooms for the course.

</div>

In this lab, we will learn how to use large language models (LLMs) – the same kind that power today’s AI assistants, but running locally on our own computers (...or the lab computers).  We will again be using the [HuggingFace transformers](https://huggingface.co/docs/transformers/index) library for this, together with open LLMs from the [HuggingFace Model Hub](https://huggingface.co/models).

In [1]:
from liu_lt import check_solutions, find_file
from liu_lt.torch import check_for_model

from transformers import pipeline, TextStreamer, AutoTokenizer

## Gemma 3 models

Concretely, we will work with LLMs from the [Gemma 3 family](https://deepmind.google/models/gemma/gemma-3/), a family of models trained to support “over 140 languages” and available in different versions and sizes.  We are using the 4B parameter version of the models, which are about 10 GB in size.  We’ll test and compare two different variants of the Gemma&nbsp;3 4B model:

- The “base” or “pre-trained” model (`pt`), which was only trained on language modelling (i.e. predicting the next word) without any further training or fine-tuning.
- The “instruction fine-tuned” model (`it`), which additionally underwent the training to turn it into an assistant model rather than “just” a next-word predictor.

The following cell will check if you’re on the lab computers and/or if your hardware seems to be sufficient for this lab:

In [2]:
it_model, pt_model = check_for_model("google/gemma-3-4b-it", "google/gemma-3-4b-pt")

['google/gemma-3-4b-it', 'google/gemma-3-4b-pt']


## Loading the tokenizer

Before we try out the “actual” LLM, we will explore how its _tokenizer_ works.  The following cell only loads Gemma&nbsp;3’s tokenizer:

In [4]:
tokenizer = AutoTokenizer.from_pretrained(pt_model)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-4b-pt.
401 Client Error. (Request ID: Root=1-69f0c91f-75be7bc03d6933b475a5b812;5878c65f-c39d-48ac-a9c7-0dd617d9d601)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-4b-pt/resolve/main/config.json.
Access to model google/gemma-3-4b-pt is restricted. You must have access to it and be authenticated to access it. Please log in.

We can call the `tokenizer` object with any string of text:

In [ ]:
tokenizer("Linköping University is a public university in Sweden.")

This shows us what the text input “looks like” to a language model at the most basic level — a series of numbers.  It’s not very useful for _analyzing_ the tokenization though.  Luckily, it’s easy to convert these numbers (the `input_ids` here) back to the actual _tokens_ they represent:

In [ ]:
input_ids = tokenizer("Linköping University is a public university in Sweden.").input_ids
tokenizer.convert_ids_to_tokens(input_ids)

If we’re just interested in the tokenization, and not the IDs, we can even use the `.tokenize()` function to save us some work:

In [ ]:
tokenizer.tokenize("Linköping University is a public university in Sweden.")

Let’s try the same in Swedish:

In [ ]:
tokenizer.tokenize("Linköpings universitet är ett statligt universitet i Sverige.")

We can see a few interesting things here:

- In the English sentence, _Linköping_ is tokenized as `Link ö ping`, whereas in the Swedish sentence, it is _Linköpings_ (with an 's') which gets tokenized as `Link öp ings`.  Small changes to a word, like adding the “s”, can result in different “splits” after tokenization!
- In the English sentence, all other words are represented by a single token, but in the Swedish sentence, _statligt_ gets split up into two (`▁stat ligt`).

## Task 1: Create your own parallel dataset

Let’s analyze the differences in tokenization a bit more systematically by creating our own small, parallel dataset.

A **parallel dataset** is a dataset consisting of sentence pairs in two languages, where one sentence is the direct translation of the other.  For example, if the two languages are English and Swedish, the two example sentences we used above could be a possible sentence pair, since they are direct translations of each other.

Your first task here is simply to compile your own, small parallel dataset, with one language being English, the other being **a language of your choice**.  You will use this small dataset to analyze (i) how the tokenizer splits up words in these two languages and if there’s a noticeable difference between them (Task&nbsp;2), and (ii) to test if the Gemma models are able to translate the sentences from your chosen language into English (Tasks&nbsp;3 and&nbsp;4).

<div class="alert alert-info">

  **Compile your own dataset of at least _ten parallel sentence pairs_ in English and one other language of your choice!** Note:

  - The “other language” can of course be Swedish, but doesn’t have to be – you could pick German, Spanish, Finnish, Greek, Arabic, ...
  - The only **requirement** for the “other language” is that **it should use spaces** to separate words – that means that e.g. Japanese or Chinese will unfortunately not work for this task.
  - You can use your own language knowledge to produce the translations, or you can use a translation tool to help you.
  - Include some **variety** in your sentences: have some shorter and some longer ones, and have some with very common expressions and some with less common/more technical vocabulary.
</div>

In [ ]:
# A list with your sentences in English – don’t change this variable name!
sentences_en = []
# A list with your sentences in the other language – don’t change this variable name either!
sentences_other = []


# YOUR CODE HERE

sentences_en = [
  "The cat sits on the windowsill.",
  "I would like a cup of coffee, please.",
  "The university is located in the city center.",
  "She enjoys reading historical novels.",
  "We will meet at eight o'clock tomorrow.",
  "The experiment produced interesting results.",
  "Can you help me with this problem?",
  "He traveled to three different countries last year.",
  "My favorite subject in school was mathematics.",
  "The weather is unusually warm for this time of year.",
]

sentences_other = [
  "El gato se sienta en el alféizar de la ventana.",
  "Quisiera una taza de café, por favor.",
  "La universidad está situada en el centro de la ciudad.",
  "A ella le gusta leer novelas históricas.",
  "Nos encontraremos a las ocho mañana.",
  "El experimento produjo resultados interesantes.",
  "¿Puedes ayudarme con este problema?",
  "Viajó a tres países diferentes el año pasado.",
  "Mi asignatura favorita en la escuela era matemáticas.",
  "El clima es inusualmente cálido para esta época del año.",
]

### 🤞 Test the tokenizer

If you produced the two lists correctly, the following cell should print the tokenized versions of each sentence pair, separated by dashes (`---`).  Inspect the tokenization to see which words have been split into multiple subword tokens – you will need this later for the reflection in Task&nbsp;2.

In [ ]:
print("---")
for en, other in zip(sentences_en, sentences_other, strict=True):
    print(" ".join(tokenizer.tokenize(en)))
    print(" ".join(tokenizer.tokenize(other)))
    print("---")

## Task 2: Compute and compare fertility scores

Look at the tokenization for your example sentences that you produced above.  Some words will get “its own token”, while others will get split up into multiple tokens.  Is there a difference between the two languages in terms of how often words seem to get split into smaller units?  Let’s analyze this a bit more systematically.

The **fertility score** is defined as the **average number of tokens per word** that the tokenizer produces on a given dataset.  Here, we use a very simplified definition of “word” as “anything that’s separated by a whitespace character”.   This means that in order to compute the fertility on a single sentence, you would take the _number of tokens_ and divide it by the _number of (whitespace-separated) words_ in that sentence.

<div class="alert alert-info">

  **Complete the function below to compute fertility!** Note:

  - The function should take a _list of tokens_ as produced by the Gemma&nbsp;3 tokenizer, and return a single number (=&nbsp;the fertility score).
  - This means that you will need to determine the _number of words_ from the list of tokens.  For this, you can make use of the fact that the tokenizer uses a special character to mark spaces (`▁`).  Note that while this looks very similar to an underscore (`_`), it is a different character!  (If you want to check for this character in your code, the easiest way is to just copy-and-paste it.)
</div>

In [ ]:
def fertility(tokens: list[str]) -> float:
    """Compute the fertility score from a list of tokens."""
    if not tokens:
        return 0.0

    # Count word starts: tokens beginning with "▁", plus the first token if it doesn't
    word_starts = sum(1 for t in tokens if t.startswith("▁"))
    if not tokens[0].startswith("▁"):
        word_starts += 1

    return len(tokens) / word_starts

### 🤞 Test your code

You can use the cell below to check if your function appears to work correctly:

In [ ]:
with check_solutions() as check:
    check.isclose(fertility(["I", "▁am", "▁fine"]), 1)
    check.isclose(fertility(["Hello", ",", "▁how", "▁are", "▁you", "?"]), 1.5)
    check.isclose(fertility(["De", "oxy", "rib", "onucle", "ic", "▁acid", "▁is", "▁a", "▁polymer", "."]), 2)

### Compare the scores

If your function works correctly, the following cell will compute the average fertility scores for the parallel dataset you created in Task 1:

In [ ]:
def average(l):
    return sum(l) / len(l)

fertility_en = average([fertility(tokenizer.tokenize(ex)) for ex in sentences_en])
print(f"Average fertility on English: {fertility_en:.2f}")
fertility_other = average([fertility(tokenizer.tokenize(ex)) for ex in sentences_other])
print(f"Average fertility on 'Other': {fertility_other:.2f}")

<div class="alert alert-info">
  <strong>Write a brief reflection on your experiments so far.</strong>  Use the following prompts:

  - How do you interpret the fertility scores you got above?
  - Looking at the actual tokens (from the end of Task&nbsp;1), for words that got split up into multiple subword tokens, do the tokens correspond to “meaningful parts” of the words or do they seem arbitrary?
</div>

The fertility score measures average number of subword tokens per whitespace separated word. A higher fertility means words are split into more subword units on average. If the Other language has a higher fertility than English, its words are segmented more often, which increases sequence length and can affect model efficiency and cost. Inspecting token splits shows many cases where subwords match meaningful prefixes/suffixes, but there are also arbitrary fragments like arbitrary splits can affect interpretability and translation quality.

SyntaxError: unterminated string literal (detected at line 7) (4077244321.py, line 7)

## Loading the base model

Now, let’s actually load the entire model.  We’re using a Huggingface [pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines) again, as it’s the most convenient way to use a model.

<div class="alert alert-warning">

⚠️ **If you run this on your own computer,** the first time you run this cell, it will automatically **download the model (ca.&nbsp;10&nbsp;GB)**.  The same applies later in this notebook when we use the instruction fine-tuned model, meaning you will need to **download ca.&nbsp;20&nbsp;GB** worth of models in total. This means you will need an internet connection and a bit of patience.

</div>

In [ ]:
# This instantiates a text generation pipeline
pt_pipe = pipeline("text-generation", model=pt_model)

# This defines how many new tokens to generate at most
pt_pipe.generation_config.max_new_tokens = 128
pt_pipe.generation_config.max_length = None  # required

Let’s try it with some simple input just to see how it works.  Before you run the following cell, think about what kind of output you would expect!

In [ ]:
pt_pipe("Who are you?")

You should notice a few things:

- The pipeline returns a list with a dictionary, containing the key `'generated_text'` with the generated text.
- The “prompt” that we gave is included in the generated text.
- The generated text might “cut off” in the middle of a sentence, as we have set a token limit of 100 new tokens to generate.

If we mainly want to play around with a model and inspect its output, there’s a more convenient way to show the generated text – adding a [TextStreamer object](https://huggingface.co/docs/transformers/main/generation_features#streaming).  This will make the text appear token by token while it is being generated, just like we see on most commercial AI assistants:

In [ ]:
streamer = TextStreamer(pt_pipe.tokenizer)
pt_pipe("Who are you?", streamer=streamer)

You might notice that running the cell above gave you **a different response** from the first time.  That’s because by default, a text generation pipeline _samples_ tokens from the vocabulary based on the learned probability distribution by the model.  In other words, the text that is generated is _not deterministic_, but may be different every time you run the generation.

**Try generating new text a few times** to see what happens!

Here’s another example you can try – or replace the prompt with your own:

In [ ]:
pt_pipe("The capital of Sweden is", streamer=streamer)

## Task 3: Get the base model to translate a sentence

Remember that the base model is only trained on language modelling.  It is _not_ trained to “follow instructions”, so we cannot prompt it with, say, “Translate this sentence for me!”, and expect this to work particularly well.

We can, however, still try to get the model to perform a task for us.  To achieve this, we need to formulate a prompt in such a way that the desired answer is a logical continuation of what came before.  This is essentially **few-shot prompting**:  We will create a prompt that consists of a few examples of what we want the model to generate, and at the end we provide it with the example we actually want it to solve, and hope that it will continue the text with the desired answer.

You will probably want to write **multi-line prompts** for this – it’s useful to know that in Python, you can write strings with line breaks in them if you surround them with triple-quotes (`"""`) rather than just a single quote.

Here is an example illustrating both a multi-line prompt and the few-shot prompting idea:


In [ ]:
pt_pipe("""The first letter in the alphabet is A.
The second letter in the alphabet is B.
The third letter in the alphabet is""", streamer=streamer)

<div class="alert alert-info">

  **Write and run a prompt below to translate a single sentence from your chosen language (from Task&nbsp;1) into English!** Note:

  - You can re-use the sentence pairs you produced in Task 1 for this, if you want.
  - The only requirements for this task are that:
    - It's clear from the prompt which sentence you want the model to translate.
    - The expected translation into English is **not** already part of the prompt (that would be “cheating”...).
    - The _initial output_ of the model after the prompt is a reasonable attempt at a translation of the sentence that you gave it.  It is likely that the model will _generate unnecessary extra output_ afterwards – we just ignore that here!
</div>

In [ ]:

try:
  prompt = f"""Translate the following sentence into English.
Spanish: {sentences_other[0]}
English:"""
  pt_pipe(prompt, streamer=streamer)
except NameError:
  print("pt_pipe or streamer not available in this environment. Run the earlier cells that load the base model.")

## Loading the instruction fine-tuned model

For the rest of this lab, we will be working with the **instruction fine-tuned** (`it`) version of the model.  To avoid memory issues, the following cell tries to “unload” the base model we used so far and free the memory:

In [ ]:
import gc, torch

for var_name in {'it_pipe', 'pt_pipe', 'tokenizer', 'streamer'}:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

<div class="alert alert-warning">

⚠️ **If you still run into memory issues** (e.g. out-of-memory errors when you load the model below), do the following:

1. Restart the Python kernel.  You can do this in Jupyter Lab by going the menu “Kernel” followed by “Restart kernel...”.
2. Run the first two code cells from this notebook which import the libraries and find the models.
3. Run the cell containing your solution from Task 1.
4. Then continue from here.
</div>

Loading the instruction fine-tuned model works exactly the same way as before, we just replace the name/path of the model (i.e. changing from `pt_model` to `it_model`):

In [ ]:
# This instantiates a text generation pipeline
it_pipe = pipeline("text-generation", model=it_model)
streamer = TextStreamer(it_pipe.tokenizer)

# This defines how many new tokens to generate at most
it_pipe.generation_config.max_new_tokens = 128
it_pipe.generation_config.max_length = None  # required

We _could_ call the model the same way as we did with the base model, but since this model is instruction fine-tuned, it is trained to expect a certain **prompt template**, i.e., a prompt that is formatted in a specific way.  For example, instead of giving the prompt as a simple text string like...

    How are you?

...the Gemma 3 IT models are trained on a chat template that looks like this:

    <bos><start_of_turn>user
    How are you?<end_of_turn>
    <start_of_turn>model

There are special symbols for (i) the beginning of the conversation (`<bos>`), (ii) the beginning and end of the user’s message (`<start_of_turn>user` ... `<end_of_turn>`), and (iii) the beginning of the model’s response (`<start_of_turn>model`).  This way, the prompt _structures_ the text like a conversation between the user and the model, just like a chat with an AI assistant!

- - - 

In Huggingface, we don’t need to construct this chat template ourselves – the tokenizer does this for us!  We just need to make sure to specify our prompt in **message format**.  A _message_ is dictionary consisting of two keys:

- The `role` specifies where the text comes from, e.g. the _user_ or the _model_.
- The `content` specifies the text itself.

For example, if we wanted to prompt the model with the same demo prompt we used above for the base model, we could write:

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"}
]

**Before you run the pipeline** on this input, think about what response you would expect this time, i.e., will this model behave similarly to or differently from the base model? Check below:

In [ ]:
it_pipe(messages, streamer=streamer)

You should see that the pipeline still returns a list with a dictionary with the `generated_text` key, but now the `generated_text` is not simply a text string, but also a _list of messages_.  The response from the model is given as a _message_ with `'role': 'assistant'`, so we know that this part comes from the model!

Can we ask the model factual knowledge questions? Let’s try!

In [ ]:
it_pipe([
    {"role": "user", "content": "What is the capital of Sweden?"}
], streamer=streamer)

This should have worked, but you will probably have gotten quite a long response in return.  If we want to _steer_ the model toward’s a _desired behaviour_, like only returning the answer itself and nothing else, we can now try to do this by _adding instructions_ to the prompt:

In [ ]:
it_pipe([
    {"role": "user", "content": "What is the capital of Sweden? Answer with a single word."}
], streamer=streamer)

Remember that you might get slightly different answers every time you run these prompts – you can try running each prompt again a few times to see what happens!

## Task 4: Using the instruction fine-tuned model for translation

Let’s try the machine translation task again, but since we’re now using a proper instruction fine-tuned model, we’ll do this in a more systematic way, and on *all* of the sentence pairs you created in Task&nbsp;1.

We will also no longer use examples, but instead try **zero-shot prompting**, which means just giving the model instructions on what we want it to do, but without giving specific _examples_ (like the example translations we used in Task&nbsp;3).

<div class="alert alert-info">

  **Complete the function below to translate a single sentence from your chosen “other” language _into_ English!** Note the following:

  - The function should take a single sentence in your chosen language (as a string) and return a string containing _only_ the translated, English sentence.
  - In this function, you should call `it_pipe` to prompt the LLM to produce the desired translation.  This means you will need to:
      1. construct an appropriate prompt, containing the instruction for the model as well as the input sentence you want it to translate, and
      2. extract the model’s response from the return value of `it_pipe()` to return it.
  - This works best if you formulate the prompt instructions in such a way that the model _only_ returns the translated English sentence, with no extra output.
  - Note that you _don't_ want the TextStreamer here, so just leave out the `streamer=streamer` argument when calling the pipeline.
</div>

In [ ]:
def translate(sentence: str) -> str:
    """Translate a single sentence from your chosen language INTO English.

    The function takes a single sentence (as a string) and should return ONLY the translated, English sentence (as a string).
    """
# YOUR CODE HERE
    # instruction and call the pipeline
    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following sentence into English. Return ONLY the translated"
                " sentence with no additional text:\n\n" + sentence
            ),
        }
    ]

    res = it_pipe(messages)

    out = res[0].get("generated_text", "")

    if isinstance(out, list):
        for m in reversed(out):
            if isinstance(m, dict) and m.get("role") == "assistant":
                return m.get("content", "").strip()
        
        return " ".join(str(x) for x in out).strip()

    return str(out).strip()

### 🤞 Test your code

To check your code, you can run the following cell, which should pass the (simple) output check, and show you the first sentence from your parallel dataset including the model’s translation:

In [ ]:
with check_solutions() as check:
    # Translate the first sentence from your “other” language
    translation = translate(sentences_other[0])
    check(isinstance(translation, str), "The translate() function should return a string")

print(f"         Original: {sentences_other[0]}")
print(f" Your translation: {sentences_en[0]}")
print(f"Model translation: {translation}")

If the output above looks sensible, you can now try running this on your entire dataset:

In [ ]:
print("---")
for en, other in zip(sentences_en, sentences_other):
    print(f"         Original: {other}")
    print(f" Your translation: {en}")
    print(f"Model translation: {translate(other)}")
    print("---")

<div class="alert alert-info">
  <strong>Write a brief reflection on your translation experiments.</strong>  Use the following prompts:

  - How difficult was it to get the model to following your instructions?  Did it reliably generate a single English sentence?
  - How good would you say the model’s translations are?  Did it manage to produce reasonably good translations?  Were there cases where it produced something clearly wrong or nonsensical?
</div>

Instruction following: The instruction tuned model generally follows the directive to return only the translated English sentence, with minimal extra output. However, some responses may include explanations or multiple sentences. Extracting just the first grammatically complete clause often yields the intended translation.

Translation quality: The model produces reasonably accurate translations that preserve meaning and grammar for most sentence pairs. Common errors include occasional word reordering, missing function words, or misalignment of idioms. The translations are recognisable and capture the core meaning, though some technical phrases may be mishandled.

## Task 5: Answering multiple-choice questions

For the final part of this notebook, we won't use any new concepts or models, but simply try to solve a different kind of task with the instruction fine-tuned model – answering multiple-choice questions.  This requires a bit more work when preparing the prompt, and we will use a different (more efficient) technique for feeding the data into the model, so that it runs faster on a dataset containing a few hundred examples.  On the other hand, this could be similar to what you might want to investigate for your course project!

The data we are using for this comes from the [AI2-ARC dataset](https://huggingface.co/datasets/allenai/ai2_arc) containing “grade-school level, multiple-choice science questions”.  Specifically, we are going to use the subset of “easy” questions, and only the validation set of this data, consisting of a total of 570&nbsp;questions.  The dataset is hosted on Huggingface, so we can use the [Huggingface Datasets library](https://huggingface.co/docs/datasets/index) to load it:

In [ ]:
from datasets import load_dataset

# This requires an internet connection (and works similarly for any other dataset on Huggingface!)
ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="validation")

### If you don’t have an internet connection or the above doesn't work for some reason, uncomment and run the following line:
# ds = load_dataset("json", data_files=find_file("ai2-arc-easy.val.jsonl"), split="train")

ds

The dataset object can be _indexed_ like a Python list.  Let’s look at the first multiple-choice question in the dataset:

In [ ]:
ds[0]

You can see that each example in the dataset is a _dictionary_ consisting of the following keys:

- `id` contains the question ID; we are not going to use this here.
- `question` contains the actual question.
- `choices` contains _two dictionaries_, where `text` is a list of the possible answers, and `label` is a list of the corresponding labels.
  - Note that the labels here are _A, B, C, D,_ but there are some questions that use e.g. _1, 2, 3, 4_ instead!
- `answerKey` contains the _label_ of the answer that should be considered correct.

To test how good our LLM is at answering these questions, we first need to **convert these questions into a prompt**, similarly to what we did in Task&nbsp;4.  Our goal here is to formulate the prompt in such a way that the LLM _only responds with the predicted answer label_ (e.g. "A"), without any additional explanations or other added text.  That way, we should be able to evaluate its performance later by comparing its response against the `answerKey` from the dataset.

<div class="alert alert-info">

  **Complete the function below to convert a single multiple-choice question into a prompt for the LLM!** Note the following:

  - The function should take a dictionary as input, in the format that you saw above for `ds[0]`.
  - The function should create a prompt containing the question, the possible answers, as well as instructions that steer the models towards responding _only_ with the desired answer label.  That means it should generally _not_ repeat the entire answer, but only output the label (e.g., "A" or "1").  Of course, make sure the prompt does **not** contain information about what the correct answer is!
  - The prompt should be added to the input dictionary under the `'prompt'` key, and the function should return the entire dictionary.  _This part is already given in the cell below!_
  - Do _not_ prompt the LLM yet!  We will do this below in a separate step.  There is a code cell below for you to quickly try out the prompt you create here.
</div>

In [ ]:
def add_prompt(example):
    """Convert a single input example from the dataset into a prompt for the LLM, and add it to the input example."""
    
    question = example.get("question", "")
    choices = example.get("choices", {}).get("text", [])
    labels = example.get("choices", {}).get("label", [])

    prompt_lines = [f"Question: {question}", "\nOptions:"]
    for lab, txt in zip(labels, choices):
        prompt_lines.append(f"{lab}: {txt}")

    prompt_lines.append("\nInstruction: Choose the correct answer and respond with ONLY the answer label (for example: A). Do not add any explanation.")

    prompt = "\n".join(prompt_lines)
    example["prompt"] = prompt
    return example

### 🤞 Test your code

To check your code, you can run the following cell, which should pass the (simple) output check, show you the prompt you created for the first example, and run it through the LLM to show you how it responds:

In [ ]:
with check_solutions() as check:
    # Add a prompt to the first example from the dataset
    result = add_prompt(ds[0])
    check(isinstance(result, dict), "The add_prompt() function should return a dictionary")
    check(result.keys() == {'id', 'question', 'choices', 'answerKey', 'prompt'}, "The returned dictionary should contain all the keys from the original example, plus a 'prompt' key")
    check(isinstance(result["prompt"], str), "The 'prompt' added to the dictionary should be a single string")

# Run this example through the LLM
it_pipe(result['prompt'], streamer=streamer)

### ⚙️ Running the model on the entire dataset

**Once you are happy with how the prompt looks and how the model responds,** you can run the following cell to add prompts to your entire dataset and run them through the LLM.  This might take a little while!

In [ ]:
# Apply the prompt to the entire dataset
ds = ds.map(add_prompt)

# Generate a list with 'messages' for every multiple-choice question
messages_list = [
    [{"role": "user", "content": prompt}] for prompt in ds["prompt"]
]

# Run this through the model in batches – this will normally be much faster than running "one example at a time"!
results = it_pipe(messages_list, batch_size=32)

Once the cell above has finished running, `results` should contain the responses from the model, in the exact same format we saw in Task&nbsp;4.

In [ ]:
print(f"Got {len(results)} results.  First result:")
results[0]

## Task 6: Evaluate the model with accuracy

For the final task, we just take all the generated responses from Task&nbsp;5 and compare them against the correct answer (the `answerKey`) in the dataset to compute a simple **accuracy score** for the model.

<div class="alert alert-info">

  **Complete the function below to compute the model’s accuracy on the question answering task!** Note the following:

  - The function takes the list of examples _(= the entire dataset)_ and the list of results _(= what we generated in the previous step)_.  Both should have the exact same length, and the _n_-th example will always correspond to the _n_-th result.  This means using the `zip()` function is a good idea here again!
  - To extract the model’s response from the “result”, you can re-use the code you wrote for solving Task&nbsp;4 (where you extracted the translation).
  - If you want to process the model’s response further, e.g. by stripping extra whitespaces or line breaks that the model produced, that’s totally okay to do here!
  - The function should then count how many of the generated answers are correct and return an accuracy score between 0 and 1.
</div>

In [ ]:
def compute_accuracy(examples, results) -> float:
    """Compare each example against the generated answer and return an accuracy score (as a number between 0 and 1)."""
    import re

    def extract_text(result):
        out = result.get("generated_text", "")
        if isinstance(out, list):
            for m in reversed(out):
                if isinstance(m, dict) and m.get("role") == "assistant":
                    return m.get("content", "").strip()
            return " ".join(str(x) for x in out).strip()
        return str(out).strip()

    total = 0
    correct = 0
    for ex, res in zip(examples, results):
        total += 1
        pred_text = extract_text(res)
        # first alphanumeric token 
        m = re.search(r"([A-Za-z0-9]+)", pred_text)
        pred_label = m.group(1).upper() if m else pred_text.strip().upper()

        gold = str(ex.get("answerKey", "")).strip().upper()
        if pred_label == gold:
            correct += 1

    return correct / total if total > 0 else 0.0

### 🤞 Test your code

You can run the following cell, which should show you the accuracy score returned by your function:

In [ ]:
compute_accuracy(ds, results)

<div class="alert alert-info">
  <strong>Final reflection!  Write a brief reflection on your multiple-choice QA experiments.</strong>  Use the following prompts:

  - How difficult was it to get the model to following your instructions?  Did it reliably generate a single label as the answer?  **Inspect a few generated results _(e.g., the first ten)_** to check this.
  - How do you interpret the accuracy score that you got?  Is it better than e.g. random guessing?
</div>

Following instructions: The model follows the instruction to output only a label, though sometimes it may generate explanatory text afterward. By extracting the first alphanumeric token, we can isolate the predicted label. A higher proportion of results that are pure labels indicates better instruction following.

Accuracy interpretation: Random guessing on a 4 choice question would yield around 25% accuracy. If the model achieves 70% accuracy it demonstrates genuine reasoning ability beyond chance. This shows the tuned LLM can not only follow instructions but also apply knowledge to answer grade school science questions reasonably well.

The instruction-tuned model  followed the instructions. In most cases, it returned only the answer label, like A, as required, although occasionally models like this may include extra text like “The answer is A”. This shows that wording of the prompt like “Output ONLY label and nothing else” is important for controlling format of the output. If needed, evaluation logic should also be able to handle small variations.

The accuracy score is 0.9, which is significantly higher than random guessing. This indicates that the model is not guessing randomly but is able to understand the question and use reasoning or knowledge to select the correct answer. Compared to the expected range of around 60–80% for a smaller model, this performance is strong.

Overall, the model performs well on this task, especially for easier questions. However, errors may still occur when questions are ambiguous, require deeper reasoning, or when multiple answer choices seem plausible. This shows that while instruction-tuned models are powerful, they are not perfect and still benefit from careful prompt design and evaluation.

In [ ]:
# inspecting first 10 results
import re
print("Inspecting first 10 model predictions:")
print("-" * 80)
for i in range(min(10, len(results))):
    res = results[i]
    ex = ds[i]
    out = res.get('generated_text', '')
    if isinstance(out, list):
        for m in reversed(out):
            if isinstance(m, dict) and m.get('role') == 'assistant':
                pred_text = m.get('content', '').strip()
                break
    else:
        pred_text = str(out).strip()
    m = re.search(r'([A-Za-z0-9]+)', pred_text)
    pred_label = m.group(1).upper() if m else pred_text.upper()
    gold = str(ex.get('answerKey', '')).strip().upper()
    match = '✓' if pred_label == gold else '✗'
    print(f"{i+1}. Gold: {gold:3s} | Pred: {pred_label:3s} {match}")
    print(f"   Full response: {pred_text[:60]}..." if len(pred_text) > 60 else f"   Full response: {pred_text}")
    print()

- - -

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

<strong>Before you submit...</strong>

➡️ Make sure you have filled in all cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.

➡️ **Make sure the notebook runs from start to finish** without errors, for example by clicking the "⏩" button in the toolbar, or via "Run$\rightarrow$Restart Kernel and Run All Cells..." in the menu.  If this is successful, you should see a green box appear below!

</div>

In [ ]:
from liu_lt import success
success("Notebook runs from start to finish!")